# Assignment: Analyzing Financial Transactions Data from Kafka Topic `fhtw-stocks`

In this assignment, you will analyze financial transactions data coming from the Kafka topic `fhtw-stocks`. The data represents stock trade transactions, including details such as the side of the trade (buy or sell), the quantity of stocks, the stock symbol, the price, the account, and the user ID.

You will perform the following tasks:
1. Set up a Spark session and configure the necessary Kafka parameters.
2. Read the streaming data from the Kafka topic.
3. Parse the JSON data and create a structured DataFrame.
4. Perform simple analysis tasks on the data, such as calculating the total quantity of stocks traded, counting the number of trades for each stock symbol, and calculating the total value of trades for each account.

### Data Structure

The messages in the Kafka topic have the following JSON structure:

```json
{
  "side": "SELL",
  "quantity": 1587,
  "symbol": "ZVV",
  "price": 326,
  "account": "LMN456",
  "userid": "User_5"
}
```

## 0. Clear Stale Checkpoint

**Run this cell first every time before (re)starting the stream.**  
A leftover checkpoint from a previous query with a different structure causes the `There are [5] sources … now [1]` error.

In [ ]:
import shutil, os

CHECKPOINT_DIR = "/tmp/checkpoints/stock_analysis"

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print(f"Removed stale checkpoint: {CHECKPOINT_DIR}")
else:
    print("No existing checkpoint found – clean start.")

## 1. Spark Session & Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, lit, sum as _sum, count, avg,
    round as _round
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

spark = SparkSession.builder \
    .appName("KafkaStockTransactionsAnalysis") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0,"
        "org.postgresql:postgresql:42.7.3"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 2. Kafka Parameters

In [ ]:
kafka_params = {
    "kafka.bootstrap.servers": "46.225.20.89:9092",
    "subscribe":               "fhtw-stocks",
    "kafka.security.protocol": "PLAINTEXT",
    "startingOffsets":         "earliest",  # consume ALL existing messages in the topic
}

## 3. JSON Schema

In [ ]:
schema = StructType([
    StructField("side",     StringType(),  True),   # "BUY" | "SELL"
    StructField("quantity", IntegerType(), True),   # number of shares
    StructField("symbol",   StringType(),  True),   # stock ticker, e.g. "ZVV"
    StructField("price",    DoubleType(),  True),   # price per share
    StructField("account",  StringType(),  True),   # account ID
    StructField("userid",   StringType(),  True),   # user ID
])

## 4. Read Raw Stream from Kafka

Kafka delivers each message payload in a binary `value` column. We cast it to a string here; JSON parsing and all aggregations happen inside `foreachBatch` on the **static** per-batch DataFrame — this sidesteps `outputMode("complete")` restrictions entirely and keeps the streaming query to a single source (no stale-checkpoint mismatches).

In [ ]:
raw_df = (
    spark.readStream
         .format("kafka")
         .options(**kafka_params)
         .load()
         .select(col("value").cast("string").alias("json_str"))
)

print("isStreaming:", raw_df.isStreaming)  # must be True

## 5. PostgreSQL Connection (Azure)

Azure Database for PostgreSQL enforces SSL — `sslmode=require` is mandatory in the JDBC URL.

In [ ]:
pg_url = (
    "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com:5432/nyt_import"
    "?sslmode=require"
)
pg_props = {
    "user":     "ds25m046",
    "password": "Technikum2025!",
    "driver":   "org.postgresql.Driver",
}
PG_TABLE = "groupname_tradereports"

## 6. Create Target Table in PostgreSQL

Run once. The unified table stores all five analyses using an `analysis_type` discriminator column.

| Column | Type | Purpose |
|---|---|---|
| `analysis_type` | VARCHAR(60) | Which of the 5 analyses this row belongs to |
| `group_key` | VARCHAR(60) | Stock symbol or account ID |
| `metric_value` | NUMERIC(18,2) | The computed aggregate |
| `inserted_at` | TIMESTAMPTZ | Auto-set on insert |

In [ ]:
import psycopg2

ddl = """
CREATE TABLE IF NOT EXISTS groupname_tradereports (
    id            SERIAL PRIMARY KEY,
    analysis_type VARCHAR(60)   NOT NULL,
    group_key     VARCHAR(60)   NOT NULL,
    metric_value  NUMERIC(18,2),
    inserted_at   TIMESTAMPTZ   NOT NULL DEFAULT NOW()
);
CREATE INDEX IF NOT EXISTS idx_tradereports_analysis
    ON groupname_tradereports (analysis_type);
"""

conn = psycopg2.connect(
    host="fhtw-big-data.postgres.database.azure.com",
    port=5432,
    dbname="nyt_import",
    user="ds25m046",
    password="Technikum2025!",
    sslmode="require"
)
with conn.cursor() as cur:
    cur.execute(ddl)
conn.commit()
conn.close()
print(f"Table '{PG_TABLE}' is ready.")

## 7. Five Analysis Aggregations + foreachBatch Writer

All aggregations run *inside* `process_batch()` on the static per-batch DataFrame:

| # | Analysis | Aggregation |
|---|---|---|
| 1 | Most traded symbol by volume | `SUM(quantity)` grouped by `symbol` |
| 2 | Highest avg trade value per symbol | `AVG(quantity × price)` grouped by `symbol` |
| 3 | Total SELL trades per stock | `COUNT(*)` where `side = SELL`, grouped by `symbol` |
| 4 | Account with largest cumulative trade value | `SUM(quantity × price)` grouped by `account` |
| 5 | Avg quantity traded per symbol | `AVG(quantity)` grouped by `symbol` |

In [ ]:
def process_batch(batch_df, batch_id):
    """
    Called once per trigger interval with the raw JSON rows for that batch.
    1. Parse JSON  →  typed columns
    2. Run the 5 aggregations on the static (non-streaming) DataFrame
    3. Union all results into one frame
    4. Overwrite PostgreSQL table with latest totals
    """
    total_rows = batch_df.count()
    print(f"\n[Batch {batch_id}] Raw rows received: {total_rows}")

    if total_rows == 0:
        print(f"[Batch {batch_id}] Empty batch – skipping.")
        return

    # ── Parse JSON ────────────────────────────────────────────────────────────
    parsed = (
        batch_df
        .select(from_json(col("json_str"), schema).alias("d"))
        .select("d.*")
        .withColumn("trade_value", col("quantity") * col("price"))
        .cache()                         # reused by all 5 aggregations below
    )

    row_count = parsed.count()
    print(f"[Batch {batch_id}] Parsed rows (non-null): {row_count}")

    if row_count == 0:
        print(f"[Batch {batch_id}] All rows failed JSON parsing – skipping.")
        parsed.unpersist()
        return

    # ── 5.1  Most Traded Symbol by Volume ────────────────────────────────────
    most_traded = (
        parsed
        .groupBy("symbol")
        .agg(_sum("quantity").cast(DoubleType()).alias("metric_value"))
        .withColumn("analysis_type", lit("1_most_traded_by_volume"))
        .withColumn("group_key",     col("symbol"))
    )

    # ── 5.2  Highest Average Trade Value per Symbol ───────────────────────────
    avg_trade_value = (
        parsed
        .groupBy("symbol")
        .agg(_round(avg("trade_value"), 2).cast(DoubleType()).alias("metric_value"))
        .withColumn("analysis_type", lit("2_highest_avg_trade_value"))
        .withColumn("group_key",     col("symbol"))
    )

    # ── 5.3  Total SELL Trades per Stock ──────────────────────────────────────
    sell_trades = (
        parsed
        .filter(col("side") == "SELL")
        .groupBy("symbol")
        .agg(count("*").cast(DoubleType()).alias("metric_value"))
        .withColumn("analysis_type", lit("3_total_sell_trades_per_stock"))
        .withColumn("group_key",     col("symbol"))
    )

    # ── 5.4  Account with Largest Cumulative Trade Value ──────────────────────
    account_trade_value = (
        parsed
        .groupBy("account")
        .agg(_round(_sum("trade_value"), 2).cast(DoubleType()).alias("metric_value"))
        .withColumn("analysis_type", lit("4_largest_cumulative_trade_value"))
        .withColumn("group_key",     col("account"))
    )

    # ── 5.5  Average Quantity per Symbol ──────────────────────────────────────
    avg_qty_per_symbol = (
        parsed
        .groupBy("symbol")
        .agg(_round(avg("quantity"), 2).cast(DoubleType()).alias("metric_value"))
        .withColumn("analysis_type", lit("5_avg_quantity_per_symbol"))
        .withColumn("group_key",     col("symbol"))
    )

    # ── Union all five result sets ─────────────────────────────────────────────
    cols = ["analysis_type", "group_key", "metric_value"]
    combined = (
        most_traded            .select(*cols)
        .union(avg_trade_value    .select(*cols))
        .union(sell_trades        .select(*cols))
        .union(account_trade_value.select(*cols))
        .union(avg_qty_per_symbol .select(*cols))
    )

    result_count = combined.count()
    print(f"[Batch {batch_id}] Writing {result_count} aggregated rows to '{PG_TABLE}' ...")

    # overwrite = table always holds the latest cumulative totals
    combined.write.jdbc(
        url=pg_url,
        table=PG_TABLE,
        mode="overwrite",
        properties=pg_props
    )

    parsed.unpersist()
    print(f"[Batch {batch_id}] Done. ✓")

## 8. Start the Streaming Query

In [ ]:
query = (
    raw_df
    .writeStream
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .foreachBatch(process_batch)
    .trigger(processingTime="60 seconds")
    .start()
)

print("Streaming query started – waiting for data …")
query.awaitTermination(timeout=300)  # stops automatically after 5 minutes
query.stop()
print("Stream stopped.")

## 9. Read Results from PostgreSQL & Display

Run this cell after the stream has finished (or after calling `query.stop()`).  
Each of the 5 analyses is queried separately and displayed as a ranked pandas DataFrame.

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display, HTML

conn = psycopg2.connect(
    host="fhtw-big-data.postgres.database.azure.com",
    port=5432,
    dbname="nyt_import",
    user="ds25m046",
    password="Technikum2025!",
    sslmode="require"
)

analyses = {
    "1_most_traded_by_volume":          ("Most Traded Symbol by Volume",             "symbol",  "total_quantity"),
    "2_highest_avg_trade_value":         ("Highest Avg Trade Value per Symbol",       "symbol",  "avg_trade_value"),
    "3_total_sell_trades_per_stock":     ("Total SELL Trades per Stock",              "symbol",  "sell_trade_count"),
    "4_largest_cumulative_trade_value":  ("Account with Largest Cumulative Trade Value", "account", "cumulative_value"),
    "5_avg_quantity_per_symbol":         ("Avg Quantity Traded per Symbol",           "symbol",  "avg_quantity"),
}

for analysis_type, (title, key_col, metric_col) in analyses.items():
    df = pd.read_sql(
        """
        SELECT group_key AS {key}, metric_value AS {metric}
        FROM   groupname_tradereports
        WHERE  analysis_type = %(atype)s
        ORDER  BY metric_value DESC
        """.format(key=key_col, metric=metric_col),
        conn,
        params={"atype": analysis_type}
    )
    display(HTML(f"<h3>{title}</h3>"))
    display(df)

conn.close()